# 29. Free words against OpenStreetMap

The demo lets Methods 1 and 2 embed the question's own words for the land, and ranks places by those words on
every method when no land class fits (barns, rivers, quarries). How good is that ranking? This notebook scores
it against OpenStreetMap features on the demo grid (90 m), for words the classifier has no class for and for a
few it has, and compares it with the class-based answer the demo gives otherwise.

It needs the demo data from notebook 13 and the OpenStreetMap layers from notebook 10. Truth is an OpenStreetMap
tag's cells on the demo grid; the ranking shows the most similar share of the study area (the demo's default
2%) and, separately, exactly as many cells as the truth has (so precision there equals recall and measures the
ranking itself). Results go to `outputs/results/29_free_words.csv`.

In [1]:
import json
import os

import numpy as np
import pandas as pd
import torch

from askmap.aether_us import TextEncoder
from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.demo.serve import load_tiles

load_env()
cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
ROOT = data_root()
DEMO = ROOT / "demo"
RES = PROJECT_ROOT / "outputs" / "results"
META = json.loads((DEMO / "meta.json").read_text())
LEAVES, S = META["leaves"], META["stride"]
RES_M = cfg["grid"]["res_m"] * S
N30 = 667 * 667
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
enc = TextEncoder(META["encoder"], META["encoder_kind"], device, os.environ["HF_HOME"])
print(f"demo grid {RES_M:.0f} m, {len(LEAVES)} classes, areas {list(META['areas'])}")

/Volumes/disco2/github/ask-the-map-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1030 17:21:19.008000 38774 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 27514.95it/s]

demo grid 90 m, 17 classes, areas ['north_dakota', 'north_carolina', 'minnesota', 'georgia']


In [2]:
def osm_mask(area, name, tag):
    """An OpenStreetMap tag's cells on the demo grid (the 30 m layer from notebook 10, every S-th cell)."""
    bits = np.unpackbits(np.load(ROOT / "osm" / f"{area}_{name}.npz")[tag])[:N30]
    return bits.reshape(667, 667).astype(bool)[::S, ::S]


def scores(text, tiles):
    q = enc([text])[0]
    q = q / np.linalg.norm(q)
    out = []
    for t in tiles:
        s = np.full(t.valid.shape, -np.inf, np.float32)
        s[t.valid] = t.z.astype(np.float32) @ q
        out.append(s)
    return out


def top(sc, tiles, share=None, k=None):
    pool = np.concatenate([s[t.valid] for s, t in zip(sc, tiles, strict=True)])
    cut = np.quantile(pool, 1 - share) if share is not None else np.sort(pool)[-k]
    return [t.valid & (s >= cut) for s, t in zip(sc, tiles, strict=True)]


def prf(pred, truth):
    tp = sum(int((p & y).sum()) for p, y in zip(pred, truth, strict=True))
    npred = sum(int(p.sum()) for p in pred)
    ntrue = sum(int(y.sum()) for y in truth)
    p = tp / npred if npred else np.nan
    r = tp / ntrue if ntrue else np.nan
    return p, r, (2 * p * r / (p + r) if tp else 0.0), npred, ntrue


def class_mask(names, tiles):
    idx = [LEAVES.index(n) for n in names]
    return [t.valid & np.isin(np.argmax(t.probs, 0), idx) for t in tiles]


# OpenStreetMap truth -> the question words tried, and the class the demo would use instead (None: no class)
CASES = {
    "osm_river": (["rivers", "a river", "a river channel"], ["water"]),
    "osm_lake": (["lakes", "a lake"], ["water"]),
    "osm_farmyard": (["barns", "farm buildings", "a farmyard"], None),
    "osm_quarry": (["a quarry", "a gravel pit or quarry"], None),
    "osm_golf": (["golf course", "a golf course"], None),
    "osm_cemetery": (["cemetery", "a cemetery"], None),
    "osm_residential": (["towns", "housing", "a residential area"], ["town"]),
    "osm_wetland": (["wetlands", "a marsh"], ["wetland"]),
}
MIN_TRUTH = 20  # cells on the demo grid; below this a score is noise

In [3]:
rows = []
for area in META["areas"]:
    tiles = load_tiles(DEMO, META)[area]
    for tag, (texts, classes) in CASES.items():
        truth = [osm_mask(area, t.name, tag) & t.valid for t in tiles]
        ntrue = sum(int(y.sum()) for y in truth)
        if ntrue < MIN_TRUTH:
            continue
        if classes:
            p, r, f, npred, _ = prf(class_mask(classes, tiles), truth)
            rows.append(
                dict(
                    area=area,
                    truth=tag,
                    how="class",
                    text="+".join(classes),
                    rule="class map",
                    precision=p,
                    recall=r,
                    f1=f,
                    shown=npred,
                    true=ntrue,
                )
            )
        for text in texts:
            sc = scores(text, tiles)
            p, r, f, npred, _ = prf(top(sc, tiles, share=0.02), truth)
            rows.append(
                dict(
                    area=area,
                    truth=tag,
                    how="words",
                    text=text,
                    rule="top 2%",
                    precision=p,
                    recall=r,
                    f1=f,
                    shown=npred,
                    true=ntrue,
                )
            )
            p, r, f, npred, _ = prf(top(sc, tiles, k=ntrue), truth)
            rows.append(
                dict(
                    area=area,
                    truth=tag,
                    how="words",
                    text=text,
                    rule="top k = truth",
                    precision=p,
                    recall=r,
                    f1=f,
                    shown=npred,
                    true=ntrue,
                )
            )
df = pd.DataFrame(rows)
df.to_csv(RES / "29_free_words.csv", index=False)
pd.set_option("display.width", 200)
print(df.round(2).to_string(index=False))

          area           truth   how                   text          rule  precision  recall   f1  shown  true
  north_dakota       osm_river class                  water     class map       0.00    0.58 0.00  10990    36
  north_dakota       osm_river words                 rivers        top 2%       0.00    0.28 0.01   2984    36
  north_dakota       osm_river words                 rivers top k = truth       0.06    0.06 0.06     36    36
  north_dakota       osm_river words                a river        top 2%       0.00    0.31 0.01   2984    36
  north_dakota       osm_river words                a river top k = truth       0.14    0.14 0.14     36    36
  north_dakota       osm_river words        a river channel        top 2%       0.01    0.92 0.02   2984    36
  north_dakota       osm_river words        a river channel top k = truth       0.14    0.14 0.14     36    36
  north_dakota        osm_lake class                  water     class map       0.26    1.00 0.41  10990  2864
 

In [4]:
# The best wording per truth and area, at the budget that measures the ranking itself (top k = truth), next to
# the class-based answer: where the words win and where they do not
best = (
    df[df.rule == "top k = truth"]
    .sort_values("precision", ascending=False)
    .groupby(["area", "truth"])
    .head(1)[["area", "truth", "text", "precision", "true"]]
)
cls = df[df.rule == "class map"][["area", "truth", "precision", "f1"]].rename(
    columns={"precision": "class_precision", "f1": "class_f1"}
)
summary = best.merge(cls, on=["area", "truth"], how="left").sort_values(["truth", "area"])
summary.to_csv(RES / "29_free_words_summary.csv", index=False)
print(summary.round(2).to_string(index=False))

          area           truth                   text  precision  true  class_precision  class_f1
       georgia    osm_cemetery             a cemetery       0.03    86              NaN       NaN
     minnesota    osm_cemetery               cemetery       0.00    68              NaN       NaN
north_carolina    osm_cemetery             a cemetery       0.02    60              NaN       NaN
     minnesota    osm_farmyard         farm buildings       0.01   145              NaN       NaN
       georgia        osm_golf            golf course       0.31   384              NaN       NaN
     minnesota        osm_golf            golf course       0.57   418              NaN       NaN
north_carolina        osm_golf          a golf course       0.35   171              NaN       NaN
       georgia        osm_lake                 a lake       0.37   374             0.19      0.31
     minnesota        osm_lake                  lakes       0.09   400             0.17      0.29
north_carolina      